# Notebook 07: SD Pipeline 完整解剖

**目标**：相比 Project 3 中的手写推理，本 notebook 聚焦**单步内部**——VAE latent 长什么样、cross-attention 关注谁、UNet 每个 block 的输出有什么差异。

**前置**：L09 + Project 3 任务 A

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from diffusers import StableDiffusionPipeline, DDIMScheduler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
pipe = StableDiffusionPipeline.from_pretrained('runwayml/stable-diffusion-v1-5', torch_dtype=dtype).to(device)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.safety_checker = None
pipe.set_progress_bar_config(disable=True)
tokenizer, text_encoder = pipe.tokenizer, pipe.text_encoder
vae, unet, scheduler = pipe.vae, pipe.unet, pipe.scheduler

## 1. VAE 的 4 个 latent channels 各编码什么？

用真实图编码看。

In [ ]:
from urllib.request import urlretrieve
from torchvision import transforms

url = 'https://hf.co/datasets/huggingface/documentation-images/resolve/main/diffusers/input_image_vermeer.png'
urlretrieve(url, 'input.png')
img = Image.open('input.png').resize((512, 512))
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3)])
img_tensor = tf(img).unsqueeze(0).to(device, dtype=dtype)

with torch.no_grad():
    latent_dist = vae.encode(img_tensor).latent_dist
    latent = latent_dist.sample() * 0.18215
print(f'latent shape: {latent.shape}')
print(f'latent stats: mean={latent.float().mean():.3f}, std={latent.float().std():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(img); axes[0].set_title('Original'); axes[0].axis('off')
for c in range(4):
    chan = latent[0, c].float().cpu().numpy()
    axes[c+1].imshow(chan, cmap='RdBu_r', vmin=-3, vmax=3)
    axes[c+1].set_title(f'latent ch{c}'); axes[c+1].axis('off')
plt.tight_layout(); plt.show()
print('观察: 不同 channel 编码不同信息（亮度、色彩对比、纹理等）。具体含义模型自行学到，不是预定义的。')

## 2. VAE 重构 vs 原图——哪些细节丢失？

In [ ]:
with torch.no_grad():
    recon = vae.decode(latent / 0.18215).sample
recon_img = (recon[0].float().cpu() / 2 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()
orig_np = np.array(img) / 255

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(orig_np); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(recon_img); axes[1].set_title('VAE recon'); axes[1].axis('off')
diff = np.abs(orig_np - recon_img).mean(axis=-1)
im = axes[2].imshow(diff, cmap='hot', vmax=0.2)
axes[2].set_title('|diff|'); axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046)
plt.show()
print(f'MAE: {diff.mean():.4f}, PSNR: {-20*np.log10(diff.mean() + 1e-8):.1f} dB')

## 3. 单步去噪：t→t-1 之间发生了什么？

生成中固定一步，看 UNet 输出与 x_t 的关系。

In [ ]:
prompt = 'a futuristic city skyline'
torch.manual_seed(42)
z = torch.randn((1, 4, 64, 64), device=device, dtype=dtype)

text_in = tokenizer(prompt, padding='max_length', max_length=77, return_tensors='pt')
uncond_in = tokenizer('', padding='max_length', max_length=77, return_tensors='pt')
with torch.no_grad():
    cond_emb = text_encoder(text_in.input_ids.to(device))[0]
    uncond_emb = text_encoder(uncond_in.input_ids.to(device))[0]
text_emb = torch.cat([uncond_emb, cond_emb], dim=0)

scheduler.set_timesteps(50)
snapshots = []  # 在不同 t 保存 (z_t, eps_pred, x0_hat)

z_cur = z.clone()
for i, t in enumerate(scheduler.timesteps):
    z_in = torch.cat([z_cur, z_cur], dim=0)
    z_in = scheduler.scale_model_input(z_in, t)
    with torch.no_grad():
        eps = unet(z_in, t, encoder_hidden_states=text_emb).sample
    eps_u, eps_c = eps.chunk(2)
    eps = eps_u + 7.5 * (eps_c - eps_u)
    # 估计 x_0
    ac_t = scheduler.alphas_cumprod[t].to(dtype)
    x0_hat = (z_cur - (1 - ac_t).sqrt() * eps) / ac_t.sqrt()
    if i in [0, 10, 25, 40, 49]:
        snapshots.append({'t': t.item(), 'z': z_cur.clone(), 'x0_hat': x0_hat.clone()})
    z_cur = scheduler.step(eps, t, z_cur).prev_sample

In [ ]:
fig, axes = plt.subplots(2, len(snapshots), figsize=(3*len(snapshots), 6))
for i, s in enumerate(snapshots):
    with torch.no_grad():
        z_decoded = vae.decode(s['z'] / 0.18215).sample
        x0_decoded = vae.decode(s['x0_hat'] / 0.18215).sample
    z_img = (z_decoded[0].float().cpu() / 2 + 0.5).clamp(0,1).permute(1,2,0).numpy()
    x0_img = (x0_decoded[0].float().cpu() / 2 + 0.5).clamp(0,1).permute(1,2,0).numpy()
    axes[0][i].imshow(z_img); axes[0][i].axis('off'); axes[0][i].set_title(f"z_t  (t={s['t']})")
    axes[1][i].imshow(x0_img); axes[1][i].axis('off')
    if i == 0: axes[1][i].set_ylabel('x̂_0 (predicted)', rotation=90)
plt.suptitle('每步的 z_t（上行）与预测的 x̂_0（下行）。x̂_0 早早就接近最终结果。')
plt.tight_layout(); plt.show()

## 4. 关键发现

- x̂_0 在前几步就大致定型——粗结构很早决定
- z_t 缓慢从噪声变为图像
- 这是为什么"早期 step 调整影响大、晚期影响小"——也是 prompt engineering 中"前几步替换"的根据

## 思考题

1. 把 VAE 重构与 16-channel VAE（SD 3）对比一下，质量提升的本质是什么？
2. 在 nb 中观察 x̂_0 在前 5 步就稳定下来，这与 "prompt-to-prompt" 论文的发现有什么关联？
3. 如果在前 25 步用 prompt A、后 25 步切换到 prompt B，结果会是什么？尝试实现并观察
4. （进阶）hook cross-attention 模块，可视化每个 token 对应的 spatial attention map